# Semana 10 — Da Random Forest às Redes Neurais
## Fundamentos e preparação específica para dados industriais

Nas aulas anteriores, percorremos o CRISP-DM, realizamos EDA, preparamos dados, treinamos uma Random Forest, avaliamos métricas e analisamos séries temporais. Agora avançaremos dentro da fase de **Modelagem**: investigaremos quando uma rede neural pode aprender padrões que modelos baseados em árvores não representam da mesma forma.

> **Questão central:** como transformar medições industriais em uma representação adequada para uma rede neural, sem perder o vínculo com o problema de negócio?

### Resultados de aprendizagem

Ao concluir este caderno, você será capaz de:

- relacionar neurônios, camadas, pesos, ativações e função de perda;
- distinguir o modo de aprendizagem de uma Random Forest e de uma MLP;
- explicar por que escala, separação dos dados e prevenção de vazamento importam;
- treinar uma MLP para classificar condições de operação;
- interpretar curva de perda, matriz de confusão, precisão, recall e F1;
- comparar a rede com um baseline já conhecido, sem assumir que “mais complexo” significa “melhor”.

## 1. Continuidade do CRISP-DM

Não reiniciaremos o ciclo. O problema, as métricas e o pipeline já existem. A mudança desta aula ocorre principalmente em **Data Preparation** e **Modeling**:

| Decisão já conhecida | Consequência para a rede neural |
|---|---|
| Separar treino e teste | O teste continua intocado até a avaliação final |
| Imputar valores ausentes | A rede não opera corretamente com `NaN` |
| Codificar categorias | Entradas precisam ser numéricas |
| Avaliar custo do falso negativo | Define qual métrica merece prioridade |
| Criar janelas temporais | Permite representar histórico como atributos |
| Comparar com baseline | Mostra se a complexidade adicional trouxe valor |

Uma rede neural é uma candidata dentro do CRISP-DM, não uma substituta para o processo.

![Da Random Forest à rede neural](images/semana_10_random_forest_para_mlp.png)

## 2. O que muda da floresta para a rede?

Uma Random Forest combina decisões discretas de várias árvores. Uma MLP combina transformações numéricas sucessivas. Para um neurônio:

\[
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
\]

\[
a = f(z)
\]

Os pesos `w` indicam como cada entrada participa da transformação; `b` desloca a fronteira; `f` introduz não linearidade. Camadas sucessivas aprendem representações intermediárias. Na classificação binária, a saída costuma representar uma probabilidade.

### Vocabulário essencial

- **época:** uma passagem pelo conjunto de treino;
- **batch:** subconjunto usado em uma atualização;
- **função de perda:** quantifica o erro que o treinamento procura reduzir;
- **backpropagation:** calcula como cada peso contribuiu para o erro;
- **otimizador:** define como os pesos são atualizados;
- **overfitting:** excelente memorização do treino, mas baixa generalização.

In [ ]:
import numpy as np

# Um neurônio simples: três medições já normalizadas.
x = np.array([0.8, 1.2, -0.4])       # temperatura, vibração e corrente
w = np.array([0.7, 1.1, -0.2])
b = -0.3

z = x @ w + b
relu = max(0.0, z)
sigmoid = 1 / (1 + np.exp(-z))

print(f"Combinação linear z = {z:.3f}")
print(f"ReLU(z) = {relu:.3f}")
print(f"Sigmoid(z) = {sigmoid:.3f}")

## 3. Por que normalizar?

Temperatura pode variar em dezenas de graus, vibração em poucos milímetros por segundo e corrente em centenas de ampères. Sem padronização, uma variável de grande magnitude pode dominar o processo de otimização.

O `StandardScaler` calcula média e desvio **somente no treino**:

\[
z = \frac{x-\mu_{treino}}{\sigma_{treino}}
\]

Aplicar `fit` antes da separação causa **vazamento de dados**: informações do teste participam indiretamente do treinamento.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 900
temperatura = rng.normal(68, 8, n)
vibracao = rng.gamma(2.2, 1.1, n)
corrente = rng.normal(42, 7, n)
pressao = rng.normal(6.2, 0.7, n)

# Rótulo sintético com interação não linear e uma parcela de ruído.
risco = (
    0.09 * (temperatura - 70)
    + 0.65 * (vibracao - 3)
    + 0.05 * (corrente - 45)
    + 0.8 * ((temperatura > 76) & (vibracao > 3.4))
    + rng.normal(0, 0.55, n)
)
falha = (risco > 0.25).astype(int)

df = pd.DataFrame({
    "temperatura_c": temperatura,
    "vibracao_mm_s": vibracao,
    "corrente_a": corrente,
    "pressao_bar": pressao,
    "falha": falha,
})
df.head()

In [ ]:
df.describe().round(2)

## 4. Pipeline comparável e reproduzível

Para uma comparação justa, Random Forest e MLP recebem a mesma divisão estratificada. A MLP recebe atributos padronizados; a floresta não depende dessa padronização, mas permanece no mesmo experimento como baseline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt

X = df.drop(columns="falha")
y = df["falha"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

mlp = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(16, 8), activation="relu",
        max_iter=300, early_stopping=False,
        validation_fraction=0.2, random_state=42
    )),
])

rf = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")

mlp.fit(X_train, y_train)
rf.fit(X_train, y_train)

for nome, modelo in {"MLP": mlp, "Random Forest": rf}.items():
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    print(f"\n{nome} — ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
    print(classification_report(y_test, pred, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nome, modelo) in zip(axes, {"MLP": mlp, "Random Forest": rf}.items()):
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)
plt.tight_layout()

## 5. Lendo o treinamento, não apenas o resultado

Uma métrica final não revela se o modelo convergiu, oscilou ou começou a memorizar. O `MLPClassifier` registra a perda de treinamento. Neste primeiro experimento observamos a curva completa; depois, vale ativar `early_stopping=True`, que reserva uma parcela do treino para validação e interrompe o processo quando não ocorre melhora relevante.

In [ ]:
historico = mlp.named_steps["modelo"].loss_curve_
plt.figure(figsize=(8, 4))
plt.plot(historico, color="#7b2cbf")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.title("Curva de aprendizagem da MLP")
plt.grid(alpha=0.25)
plt.show()
print("Épocas executadas:", len(historico))

## 6. Interpretação orientada ao negócio

Considere uma parada não detectada muito mais cara que uma inspeção desnecessária. Nesse caso, o **recall da classe de falha** pode ser mais importante que a acurácia global. Alterar o limiar de decisão muda o equilíbrio entre falsos positivos e falsos negativos.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = mlp.predict_proba(X_test)[:, 1]
linhas = []
for limiar in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (proba >= limiar).astype(int)
    linhas.append({
        "limiar": limiar,
        "precisão_falha": precision_score(y_test, pred, zero_division=0),
        "recall_falha": recall_score(y_test, pred, zero_division=0),
        "f1_falha": f1_score(y_test, pred, zero_division=0),
    })
pd.DataFrame(linhas).round(3)

## 7. Atividade de investigação

1. Altere a arquitetura para `(8,)`, `(32, 16)` e `(32, 16, 8)`. Registre o número de épocas e o desempenho.
2. Retire temporariamente o `StandardScaler`. O que muda na estabilidade e no resultado?
3. Compare `relu` e `tanh` mantendo as demais decisões fixas.
4. Escolha um limiar considerando que um falso negativo custa cinco vezes mais que um falso positivo.
5. Redija uma conclusão de CRISP-DM: a MLP superou o baseline de maneira suficiente para justificar sua complexidade?

### Evidência esperada

Uma tabela com configuração, métrica técnica, custo estimado e justificativa. Não escolha o modelo apenas pela maior acurácia.

## 8. Síntese

- Redes neurais aprendem transformações sucessivas; árvores combinam regras de particionamento.
- Preparação e escala passam a influenciar diretamente o treinamento.
- A comparação deve usar a mesma divisão e métricas coerentes com o negócio.
- A curva de perda ajuda a diagnosticar a aprendizagem.
- O CRISP-DM continua sendo a estrutura de decisão: o algoritmo é apenas uma peça.

## Bibliografia e aprofundamento

- PEDREGOSA, F. et al. *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research, 2011.
- GÉRON, Aurélien. *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*. 3. ed. O'Reilly, 2022.
- GOODFELLOW, Ian; BENGIO, Yoshua; COURVILLE, Aaron. *Deep Learning*. MIT Press, 2016. Disponível em: <https://www.deeplearningbook.org/>.
- Scikit-learn. **MLPClassifier**. <https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html>.
- Scikit-learn. **StandardScaler**. <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>.
- Scikit-learn. **Common pitfalls and recommended practices**. <https://scikit-learn.org/stable/common_pitfalls.html>.